# Entrega MovieLens — pipeline acadêmico e perfil por filmes
Limpeza, matriz, treino/teste e baselines preservados. Demonstração local com busca e confirmação de títulos, histórico e até cinco referências. Interpretação limitada de comandos, sem LLM.


In [ ]:
%pip -q install "numpy==2.5.2" "pandas==3.0.5" "scipy==1.18.1" "python-dateutil==2.9.0.post0" "six==1.17.0"
# Requer Python 3.12 para este ambiente fixado.


## 1. Criar os módulos atualizados
O código é uma cópia dos módulos do projeto. A preparação e a avaliação usam exclusivamente estatísticas do treino.


In [ ]:
from pathlib import Path
Path('recomendador.py').write_text('"""Preparação, avaliação e recomendação. Python >=3.10; sem API externa."""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport io\nimport json\nimport re\nimport unicodedata\nimport urllib.request\nimport urllib.error\nimport ssl\nimport zipfile\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import csr_matrix, save_npz\n\nROOT = Path(__file__).resolve().parent\nURL = \'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip\'\nSEED = 42\n\ndef normalizar(texto):\n    return \'\'.join(c for c in unicodedata.normalize(\'NFKD\', texto.lower()) if not unicodedata.combining(c))\n\n# Vocabulário transparente: não é um modelo de linguagem nem entende toda frase.\nVOCAB = {\n \'Action\': [\'acao\', \'action\', \'luta\', \'lutas\', \'super heroi\', \'super herois\'],\n \'Adventure\': [\'aventura\', \'adventure\', \'exploracao\'],\n \'Animation\': [\'animacao\', \'animation\', \'anime\', \'desenho\'],\n \'Children\': [\'infantil\', \'criancas\', \'children\'],\n \'Comedy\': [\'comedia\', \'comedy\', \'engracado\', \'humor\'],\n \'Crime\': [\'crime\', \'crimes\', \'policial\', \'investigacao\', \'detetive\'],\n \'Documentary\': [\'documentario\', \'documentary\'],\n \'Drama\': [\'drama\', \'dramatico\'],\n \'Fantasy\': [\'fantasia\', \'fantasy\', \'magia\', \'magico\', \'sobrenatural\'],\n \'Film-Noir\': [\'noir\'],\n \'Horror\': [\'terror\', \'horror\', \'assustador\', \'medo\'],\n \'Musical\': [\'musical\', \'musicais\'],\n \'Mystery\': [\'misterio\', \'mystery\', \'enigma\', \'investigacao\'],\n \'Romance\': [\'romance\', \'romantico\', \'amor\'],\n \'Sci-Fi\': [\'ficcao cientifica\', \'sci fi\', \'scifi\', \'espaco\', \'futurista\', \'alienigena\', \'robos\'],\n \'Thriller\': [\'suspense\', \'thriller\', \'tensao\'],\n \'War\': [\'guerra\', \'war\'],\n \'Western\': [\'faroeste\', \'western\', \'velho oeste\'],\n}\n\ndef interpretar(texto):\n    t = re.sub(r\'[^a-z0-9 ]\', \' \', normalizar(texto))\n    t = re.sub(r\'\\s+\', \' \', t).strip()\n    return [g for g, palavras in VOCAB.items() if any(re.search(r\'\\b\' + re.escape(p) + r\'\\b\', t) for p in palavras)]\n\ndef carregar(pasta=None):\n    pasta = Path(pasta) if pasta else ROOT / \'dados\' / \'ml-latest-small\'\n    if not all((pasta / f).exists() for f in [\'ratings.csv\', \'movies.csv\']):\n        pasta.parent.mkdir(parents=True, exist_ok=True)\n        zip_local = pasta.parent / \'ml-latest-small.zip\'\n        if zip_local.is_file():\n            print(\'Usando ZIP local: \' + str(zip_local), flush=True)\n            arquivo = zip_local.read_bytes()\n        else:\n            print(\'Baixando MovieLens oficial...\', flush=True)\n            try:\n                with urllib.request.urlopen(URL, timeout=60) as resposta:\n                    arquivo = resposta.read()\n            except urllib.error.URLError as erro:\n                certificado = isinstance(erro.reason, ssl.SSLCertVerificationError)\n                motivo = \'O Python não conseguiu validar o certificado HTTPS.\' if certificado else \'Não foi possível baixar o MovieLens: \' + str(erro.reason)\n                raise RuntimeError(motivo + \'\\nBaixe \' + URL + \' pelo navegador e salve em \' + str(zip_local) + \'\\nDepois execute o aplicativo novamente. Também pode extrair ratings.csv e movies.csv em \' + str(pasta) + \'.\') from erro\n        # Salva a origem e o hash para identificar a versão usada no experimento.\n        (pasta.parent / \'origem.json\').write_text(json.dumps({\'url\': URL, \'arquivo_local\': str(zip_local) if zip_local.is_file() else None, \'sha256_zip\': hashlib.sha256(arquivo).hexdigest()}, indent=2))\n        with zipfile.ZipFile(io.BytesIO(arquivo)) as z:\n            for nome in [\'ratings.csv\', \'movies.csv\', \'tags.csv\', \'links.csv\', \'README.txt\']:\n                membro = \'ml-latest-small/\' + nome\n                pasta.mkdir(parents=True, exist_ok=True)\n                (pasta / nome).write_bytes(z.read(membro))\n    return pd.read_csv(pasta / \'ratings.csv\'), pd.read_csv(pasta / \'movies.csv\')\n\ndef limpar(ratings, movies):\n    r, m = ratings.copy(), movies.copy()\n    for coluna in [\'userId\', \'movieId\', \'rating\', \'timestamp\']:\n        r[coluna] = pd.to_numeric(r[coluna], errors=\'coerce\')\n    m[\'movieId\'] = pd.to_numeric(m[\'movieId\'], errors=\'coerce\')\n    m = m.dropna(subset=[\'movieId\', \'title\']).drop_duplicates(\'movieId\').copy()\n    m = m[np.isfinite(m.movieId) & m.movieId.gt(0) & m.movieId.mod(1).eq(0)].copy()\n    m[\'movieId\'] = m.movieId.astype(int)\n    m[\'genres\'] = m.genres.fillna(\'(no genres listed)\')\n    validos = np.isfinite(r[[\'userId\',\'movieId\',\'rating\',\'timestamp\']]).all(axis=1)\n    validos &= r.rating.between(.5, 5) & r.rating.mul(2).mod(1).eq(0)\n    validos &= r.userId.gt(0) & r.userId.mod(1).eq(0) & r.movieId.mod(1).eq(0) & r.timestamp.ge(0)\n    r = r[validos & r.movieId.isin(m.movieId)].copy()\n    r[[\'userId\', \'movieId\', \'timestamp\']] = r[[\'userId\', \'movieId\', \'timestamp\']].astype(\'int64\')\n    r = r.sort_values([\'timestamp\',\'userId\',\'movieId\'], kind=\'stable\').drop_duplicates([\'userId\', \'movieId\'], keep=\'last\')\n    m[\'ano\'] = pd.to_numeric(m.title.str.extract(r\'\\((\\d{4})\\)\\s*$\', expand=False), errors=\'coerce\').astype(\'Int64\')\n    return r.reset_index(drop=True), m.reset_index(drop=True), {\'ratings_entrada\': len(ratings), \'ratings_limpos\': len(r), \'ratings_removidos\': len(ratings)-len(r), \'filmes_catalogo\': len(m)}\n\ndef dividir(ratings, fracao=.2, seed=SEED):\n    """Holdout aleatório por usuário; mantém pelo menos uma avaliação no treino."""\n    if not 0 < fracao < 1:\n        raise ValueError(\'A fração de teste precisa estar entre 0 e 1.\')\n    rng = np.random.default_rng(seed)\n    teste_indices = []\n    for _, grupo in ratings.sort_values([\'userId\',\'movieId\']).groupby(\'userId\', sort=True):\n        n = min(len(grupo)-1, max(1, int(np.floor(len(grupo)*fracao)))) if len(grupo)>1 else 0\n        if n:\n            teste_indices.extend(rng.choice(grupo.index.to_numpy(), n, replace=False))\n    mask = ratings.index.isin(teste_indices)\n    return ratings.loc[~mask].copy(), ratings.loc[mask].copy()\n\ndef ranking(treino, min_filme=5):\n    if treino.empty:\n        raise ValueError(\'Treino vazio; confira os filtros.\')\n    stats = treino.groupby(\'movieId\').rating.agg(contagem=\'count\', media=\'mean\')\n    # Média global e contagens calculadas exclusivamente no treino recebido.\n    stats[\'nota_ponderada\'] = (stats.contagem*stats.media + min_filme*treino.rating.mean()) / (stats.contagem+min_filme)\n    return stats[stats.contagem >= min_filme].reset_index()\n\ndef avaliar(treino, teste, stats, k=10, relevante=4.):\n    candidatos = set(stats.movieId)\n    vistos = {u:set(g.movieId) for u,g in treino.groupby(\'userId\')}\n    linhas=[]\n    for metodo, coluna in [(\'popularidade\',\'contagem\'), (\'nota_ponderada\',\'nota_ponderada\')]:\n        ordem = stats.sort_values([coluna,\'contagem\',\'movieId\'], ascending=[False,False,True]).movieId.tolist()\n        for u,g in teste.groupby(\'userId\'):\n            positivos=set(g.loc[g.rating>=relevante,\'movieId\'])\n            if not positivos:\n                continue\n            rec=[i for i in ordem if i not in vistos.get(u,set())][:k]\n            acertos=[int(i in positivos) for i in rec]\n            dcg=sum(v/np.log2(j+2) for j,v in enumerate(acertos))\n            ideal=sum(1/np.log2(j+2) for j in range(min(k,len(positivos))))\n            linhas.append({\'metodo\':metodo,\'userId\':int(u),\'precision_at_10\':sum(acertos)/k,\'recall_at_10\':sum(acertos)/len(positivos),\'ndcg_at_10\':dcg/ideal,\'relevantes_teste\':len(positivos),\'relevantes_elegiveis\':len(positivos & candidatos)})\n    df=pd.DataFrame(linhas)\n    if df.empty:\n        return df, {\'aviso\':\'Não há usuários com notas relevantes no teste.\'}\n    metricas=df.groupby(\'metodo\')[[\'precision_at_10\',\'recall_at_10\',\'ndcg_at_10\']].mean().to_dict(\'index\')\n    base=df[df.metodo==\'popularidade\']\n    return df, {\'metricas\':metricas,\'usuarios_avaliados\':len(base),\'usuarios_sem_relevantes_excluidos\':int(teste.userId.nunique()-len(base)),\'relevantes_teste\':int(base.relevantes_teste.sum()),\'relevantes_elegiveis\':int(base.relevantes_elegiveis.sum()),\'protocolo\':\'Todos os itens elegíveis, sem amostragem de negativos. Itens relevantes fora do catálogo contam como não recuperados. Precision divide por 10 mesmo quando há menos resultados. Notas >=4 são relevantes.\'}\n\ndef preparar(pasta=None, min_filme=5, min_usuario=5):\n    bruto,m=carregar(pasta)\n    r,m,auditoria=limpar(bruto,m)\n    treino_bruto,teste=dividir(r)\n    # Elegibilidade é decidida só pelo treino. Não se usa a nota do teste.\n    bons=treino_bruto.groupby(\'userId\').size()\n    usuarios=set(bons[bons>=min_usuario].index)\n    treino=treino_bruto[treino_bruto.userId.isin(usuarios)].copy()\n    teste=teste[teste.userId.isin(usuarios)].copy()\n    stats=ranking(treino,min_filme)\n    saida=ROOT/\'resultados\'; saida.mkdir(exist_ok=True)\n    treino.to_csv(saida/\'treino.csv\',index=False); teste.to_csv(saida/\'teste.csv\',index=False)\n    m.to_csv(saida/\'filmes.csv\',index=False); stats.to_csv(saida/\'ranking_treino.csv\',index=False)\n    users=sorted(treino.userId.unique()); itens=sorted(treino.movieId.unique())\n    um={v:i for i,v in enumerate(users)}; im={v:i for i,v in enumerate(itens)}\n    matriz=csr_matrix((treino.rating.to_numpy(),(treino.userId.map(um),treino.movieId.map(im))),shape=(len(users),len(itens)))\n    save_npz(saida/\'matriz_treino.npz\',matriz)\n    (saida/\'indices_matriz.json\').write_text(json.dumps({\'userIds\':[int(x) for x in users],\'movieIds\':[int(x) for x in itens]}))\n    detalhes,resumo=avaliar(treino,teste,stats)\n    detalhes.to_csv(saida/\'metricas_por_usuario.csv\',index=False)\n    resumo.update({\'auditoria\':auditoria,\'seed\':SEED,\'fracao_teste\':.2,\'min_avaliacoes_filme_treino\':min_filme,\'min_avaliacoes_usuario_treino\':min_usuario,\'linhas_treino\':len(treino),\'linhas_teste\':len(teste),\'itens_elegiveis\':len(stats),\'matriz_shape\':list(matriz.shape),\'matriz_esparsidade\':1-matriz.nnz/(matriz.shape[0]*matriz.shape[1]),\'observacao\':\'Métricas avaliam baselines por histórico; não avaliam compreensão de texto. A matriz mantém todas as avaliações dos usuários elegíveis; o corte por filme define o catálogo dos baselines.\'})\n    (saida/\'relatorio.json\').write_text(json.dumps(resumo,ensure_ascii=False,indent=2))\n    print(json.dumps(resumo,ensure_ascii=False,indent=2))\n    return treino,teste,m,stats\n\nclass Recomendador:\n    def __init__(self, movies, stats, treino=None):\n        self.filmes=movies.drop_duplicates(\'movieId\').copy()\n        if \'ano\' not in self.filmes:\n            self.filmes[\'ano\'] = pd.to_numeric(self.filmes.title.str.extract(r\'\\((\\d{4})\\)\\s*$\', expand=False), errors=\'coerce\').astype(\'Int64\')\n        self.catalogo=self.filmes.merge(stats.drop_duplicates(\'movieId\'),on=\'movieId\',how=\'inner\')\n        self.vistos={} if treino is None else {int(u):set(g.movieId) for u,g in treino.groupby(\'userId\')}\n\n    def recomendar_perfil(self, sessao, k=10):\n        """Perfil exclusivamente de catálogo + estatísticas do treino; sem acesso ao teste."""\n        if not sessao.assistidos:\n            return []\n        def generos(valor):\n            return set(valor.split(\'|\')) - {\'(no genres listed)\'}\n        referencias = [generos(f.genres) for f in self.filmes.itertuples()\n                       if f.movieId in sessao.referencias and f.movieId not in sessao.rejeitados]\n        resultados = []\n        for f in self.catalogo.itertuples():\n            if f.movieId in sessao.assistidos:\n                continue\n            gs = generos(f.genres)\n            if gs & sessao.negativos:\n                continue\n            comuns_preferidos = gs & sessao.positivos\n            if sessao.positivos and not comuns_preferidos:\n                continue\n            similaridade = sum(len(gs & r) / len(gs | r) if gs | r else 0 for r in referencias) / len(referencias) if referencias else 0\n            if referencias and not sessao.positivos and similaridade == 0:\n                continue\n            preferencia = len(comuns_preferidos) / len(sessao.positivos) if sessao.positivos else 0\n            score = .65 * similaridade + .25 * preferencia + .10 * float(f.nota_ponderada) / 5\n            comuns = gs & set().union(*referencias) if referencias else set()\n            partes = []\n            if comuns:\n                partes.append(\'Gêneros em comum com as referências: \' + \', \'.join(sorted(comuns)))\n            if comuns_preferidos:\n                partes.append(\'Preferências atendidas: \' + \', \'.join(sorted(comuns_preferidos)))\n            if not partes:\n                partes.append(\'Sem referências ativas: ordenação pela nota ponderada do treino\')\n            partes.append(\'A nota ponderada do treino também compõe a pontuação\')\n            resultados.append({\'movieId\': int(f.movieId), \'titulo\': f.title,\n                \'ano\': int(f.ano) if pd.notna(f.ano) else None, \'generos\': f.genres,\n                \'nota_media_treino\': round(float(f.media), 2), \'avaliacoes_treino\': int(f.contagem),\n                \'motivo\': \'. \'.join(partes) + \'.\', \'pontuacao\': score,\n                \'_nota\': float(f.nota_ponderada)})\n        resultados.sort(key=lambda f: (-f[\'pontuacao\'], -f[\'_nota\'], -f[\'avaliacoes_treino\'], f[\'movieId\']))\n        for f in resultados:\n            del f[\'_nota\']\n        return resultados[:max(1, min(10, int(k))) ]\n\n    def recomendar(self,texto,k=10,user_id=None):\n        k=max(1,min(10,int(k)))\n        generos=interpretar(texto)\n        c=self.catalogo.copy()\n        if user_id is not None:\n            c=c[~c.movieId.isin(self.vistos.get(int(user_id),set()))].copy()\n        c[\'acertos\']=c.genres.map(lambda s:len(set(s.split(\'|\')) & set(generos)))\n        if generos:\n            c=c[c.acertos>0].copy()\n        c=c.sort_values([\'acertos\',\'nota_ponderada\',\'contagem\',\'movieId\'],ascending=[False,False,False,True]).head(k)\n        resultados=[]\n        for _,f in c.iterrows():\n            comuns=[g for g in generos if g in f.genres.split(\'|\')]\n            resultados.append({\'movieId\':int(f.movieId),\'titulo\':f.title,\'generos\':f.genres,\'nota_media_treino\':round(float(f.media),2),\'avaliacoes_treino\':int(f.contagem),\'motivo\':(\'Gêneros relacionados ao pedido: \'+\', \'.join(comuns)) if comuns else \'Sem gênero reconhecido: sugestão pela nota ponderada e número de avaliações.\'})\n        aviso=\'Protótipo por palavras-chave: não interpreta negações, enredos ou preferências complexas. Não informa disponibilidade em plataformas.\'\n        if not generos: aviso=\'Não reconheci gêneros no texto; usei o ranking geral. \'+aviso\n        return {\'generos_identificados\':generos,\'aviso\':aviso,\'recomendacoes\':resultados}\n\ndef carregar_modelo():\n    p=ROOT/\'resultados\'\n    if not (p/\'relatorio.json\').exists(): preparar()\n    return Recomendador(pd.read_csv(p/\'filmes.csv\'),pd.read_csv(p/\'ranking_treino.csv\'),pd.read_csv(p/\'treino.csv\'))\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--dados\',help=\'Pasta contendo ratings.csv e movies.csv\')\n    parser.add_argument(\'--min-filme\',type=int,default=5)\n    parser.add_argument(\'--min-usuario\',type=int,default=5)\n    args=parser.parse_args()\n    if args.min_filme<1 or args.min_usuario<1: parser.error(\'Os mínimos precisam ser positivos.\')\n    preparar(args.dados,args.min_filme,args.min_usuario)\n', encoding="utf-8")
Path('conversa.py').write_text('"""Interpretação local limitada, catálogo e estado explícito por sessão."""\nfrom dataclasses import dataclass, field\nimport re\nfrom recomendador import normalizar, interpretar\n\n\n@dataclass\nclass Sessao:\n    assistidos: list = field(default_factory=list)\n    referencias: list = field(default_factory=list)\n    rejeitados: set = field(default_factory=set)\n    positivos: set = field(default_factory=set)\n    negativos: set = field(default_factory=set)\n    recomendacoes: list = field(default_factory=list)\n    pendentes: set = field(default_factory=set)\n\n\ndef interpretar_mensagem(texto):\n    """Gramática deliberadamente restrita; nunca infere títulos ou IDs."""\n    t = normalizar(texto.strip()).rstrip(\'.!?\')\n    if t == \'limpar minhas preferencias\':\n        return (\'limpar\', None)\n    if t.startswith(\'por que\'):\n        return (\'explicar\', re.sub(r\'^por que (?:voce recomendou )?\', \'\', t).strip())\n    for prefixo, acao in [(\'assisti \', \'assistiu\'), (\'quero considerar \', \'considerar\')]:\n        if t.startswith(prefixo):\n            alvo = t[len(prefixo):]\n            rejeita = bool(re.search(r\',? mas nao gostei$\', alvo))\n            alvo = re.sub(r\',? mas nao gostei$\', \'\', alvo).strip()\n            return (acao, (alvo, rejeita))\n    # Cada oração precisa ter uma intenção explícita. Negação nunca vira positivo.\n    partes = re.split(r\'\\s*(?:[;.!]|\\s+e\\s+(?=nao gosto|quero|prefiro|gosto))\\s*\', t)\n    preferencias = []\n    for parte in partes:\n        if not parte:\n            continue\n        m = re.fullmatch(r\'(nao gosto de|quero evitar|sem|quero mais|quero|prefiro|gosto de)\\s+(.+)\', parte)\n        if not m:\n            return (\'desconhecida\', None)\n        generos = interpretar(m[2])\n        # Rejeita qualificadores que o parser não sabe respeitar ("só", "sem", anos…).\n        from recomendador import VOCAB\n        sobra = re.sub(r\'[^a-z0-9 ]\', \' \', m[2])\n        for palavra in sorted({p for vs in VOCAB.values() for p in vs}, key=len, reverse=True):\n            sobra = re.sub(r\'\\b\' + re.escape(palavra) + r\'\\b\', \' \', sobra)\n        sobra = re.sub(r\'\\b(e|de|com)\\b\', \' \', sobra)\n        if not generos or sobra.strip():\n            return (\'desconhecida\', None)\n        preferencias.append((m[1] in (\'nao gosto de\', \'quero evitar\', \'sem\'), generos))\n    return (\'preferencias\', preferencias) if preferencias else (\'desconhecida\', None)\n\n\nclass Conversa:\n    def __init__(self, modelo):\n        self.modelo = modelo\n        self.filmes = modelo.filmes.set_index(\'movieId\', drop=False).rename_axis(\'catalogo_id\')\n\n    def filme(self, movie_id):\n        f = self.filmes.loc[movie_id]\n        return {\'movieId\': int(movie_id), \'titulo\': f.title,\n                \'ano\': int(f.ano) if str(f.ano) not in (\'<NA>\', \'nan\') else None,\n                \'generos\': f.genres}\n\n    def buscar(self, consulta):\n        q = normalizar(consulta.strip())\n        if not q:\n            return []\n        titulos = self.filmes.title.map(normalizar)\n        # Ano omitido não desfaz a ambiguidade de remakes.\n        exato = titulos.str.replace(r\'\\s*\\(\\d{4}\\)\\s*$\', \'\', regex=True).eq(q) | titulos.eq(q)\n        achados = self.filmes[exato]\n        if achados.empty:\n            achados = self.filmes[titulos.str.contains(q, regex=False)]\n        return [self.filme(i) for i in achados.sort_values([\'title\', \'movieId\']).index[:30]]\n\n    def resposta(self, s, mensagem):\n        return {\'mensagem\': mensagem, \'assistidos\': [self.filme(i) for i in s.assistidos],\n                \'referencias\': [self.filme(i) for i in s.referencias],\n                \'excedentes\': [self.filme(i) for i in s.assistidos if i not in s.referencias and i not in s.rejeitados],\n                \'rejeitados\': [self.filme(i) for i in s.assistidos if i in s.rejeitados],\n                \'preferencias\': {\'positivas\': sorted(s.positivos), \'negativas\': sorted(s.negativos)},\n                \'recomendacoes\': s.recomendacoes}\n\n    def atualizar(self, s):\n        s.recomendacoes = self.modelo.recomendar_perfil(s)\n        return (\' Há apenas \' + str(len(s.recomendacoes)) + \' candidatos adequados; mantive suas restrições.\') if len(s.recomendacoes) < 10 else \'\'\n\n    def considerar(self, s, movie_id, substituir=None):\n        if type(movie_id) is not int or movie_id not in s.assistidos:\n            raise ValueError(\'Adicione e confirme esse filme primeiro.\')\n        if movie_id in s.rejeitados:\n            raise ValueError(\'Esse filme foi rejeitado. Ele permanece apenas no histórico.\')\n        if movie_id in s.referencias:\n            return \'Esse filme já é uma referência ativa.\'\n        if len(s.referencias) == 5:\n            if substituir not in s.referencias:\n                return \'Escolha no controle de referências qual dos cinco filmes substituir.\'\n            s.referencias[s.referencias.index(substituir)] = movie_id\n        else:\n            s.referencias.append(movie_id)\n        return \'Referências atualizadas.\' + self.atualizar(s)\n\n    def executar(self, s, dados):\n        acao = dados.get(\'acao\', \'mensagem\')\n        if acao == \'reiniciar\':\n            s.__dict__.update(Sessao().__dict__)\n            return self.resposta(s, \'Quais filmes você já assistiu? Busque e confirme pelo menos um título. O ideal é de 2 a 5.\')\n        if acao == \'estado\':\n            return self.resposta(s, \'Quais filmes você já assistiu? Busque os títulos e confirme a seleção.\' if not s.assistidos else \'Sua sessão foi mantida.\')\n        if acao == \'buscar\':\n            texto = dados.get(\'texto\', \'\')\n            if not isinstance(texto, str) or not 0 < len(texto) <= 2000:\n                raise ValueError(\'Informe títulos separados por ponto e vírgula ou uma linha por título.\')\n            consultas = [p.strip() for p in re.split(r\'[;\\n]\', texto) if p.strip()]\n            buscas = [{\'consulta\': q, \'opcoes\': self.buscar(q)} for q in consultas]\n            s.pendentes = {f[\'movieId\'] for b in buscas for f in b[\'opcoes\']}\n            d = self.resposta(s, \'Revise os títulos e anos antes de confirmar. Se não encontrar, tente parte do título em inglês. Cada busca mostra até 30 opções; acrescente o ano para restringir.\')\n            d[\'buscas\'] = buscas\n            return d\n        if acao in (\'confirmar\', \'confirmar_selecao\'):\n            ids = dados.get(\'ids\')\n            permitidos = s.pendentes if acao == \'confirmar\' else self.filmes.index\n            if not isinstance(ids, list) or not ids or any(type(i) is not int or i not in permitidos for i in ids):\n                raise ValueError(\'Selecione pelo menos um filme da busca atual e confirme o ano.\')\n            novos = list(dict.fromkeys(ids))\n            inicial = not s.assistidos\n            for i in novos:\n                if i not in s.assistidos:\n                    s.assistidos.append(i)\n                    if len(s.referencias) < 5:\n                        s.referencias.append(i)\n            s.pendentes.clear()\n            msg = \'Filmes confirmados. Os títulos são referências provisórias, sem presumir que você gostou. Você pode dizer “assisti TÍTULO, mas não gostei”.\' if inicial else \'Histórico atualizado. Os novos filmes preencheram as referências disponíveis, até cinco; os demais podem ser considerados pelos controles.\'\n            if len(s.referencias) == 1:\n                msg += \' Uma referência deixa o perfil abrangente; informar outro filme pode melhorar as sugestões.\'\n            if len(s.assistidos) > 5:\n                msg += \' Usei inicialmente até cinco referências. Os demais estão em “Assistidos fora do perfil”, também excluídos das sugestões; você pode considerar um deles.\'\n            return self.resposta(s, msg + self.atualizar(s))\n        if acao == \'considerar\':\n            return self.resposta(s, self.considerar(s, dados.get(\'movieId\'), dados.get(\'substituir\')))\n        if acao == \'rejeitar\':\n            i = dados.get(\'movieId\')\n            if type(i) is not int or i not in self.filmes.index:\n                raise ValueError(\'Selecione um filme válido.\')\n            if i not in s.assistidos:\n                s.assistidos.append(i)\n            s.rejeitados.add(i)\n            if i in s.referencias:\n                s.referencias.remove(i)\n            return self.resposta(s, \'Retirei o filme das referências. Ele continua no histórico e não será recomendado.\' + self.atualizar(s))\n        if acao != \'mensagem\':\n            raise ValueError(\'Ação desconhecida.\')\n        texto = dados.get(\'texto\', \'\')\n        if not isinstance(texto, str) or not 0 < len(texto.strip()) <= 2000:\n            raise ValueError(\'Escreva uma mensagem de até 2000 caracteres.\')\n        if not s.assistidos:\n            return self.resposta(s, \'Para começar, busque e confirme pelo menos um filme assistido. Pode estar registrado em inglês no catálogo.\')\n        intencao, valor = interpretar_mensagem(texto)\n        if intencao == \'limpar\':\n            s.positivos.clear(); s.negativos.clear()\n            return self.resposta(s, \'Preferências de gêneros limpas; histórico, referências e rejeições mantidos.\' + self.atualizar(s))\n        if intencao == \'preferencias\':\n            for negativo, generos in valor:\n                destino, outro = (s.negativos, s.positivos) if negativo else (s.positivos, s.negativos)\n                destino.update(generos); outro.difference_update(generos)\n            return self.resposta(s, \'Preferências atualizadas. Gêneros rejeitados são excluídos.\' + self.atualizar(s))\n        if intencao in (\'assistiu\', \'considerar\'):\n            alvo, rejeita = valor\n            filmes = self.buscar(alvo)\n            # Conversa exige título exato e único. Partial só com revisão visual.\n            exatos = [f for f in filmes if normalizar(f[\'titulo\']) == alvo or re.sub(r\'\\s*\\(\\d{4}\\)$\', \'\', normalizar(f[\'titulo\'])) == alvo]\n            if len(exatos) != 1:\n                return self.resposta(s, \'Não identifiquei um título único. Use “Adicionar filmes” para buscar, revisar o ano e confirmar; suas recomendações foram mantidas.\')\n            i = exatos[0][\'movieId\']\n            if intencao == \'considerar\':\n                return self.resposta(s, self.considerar(s, i))\n            if rejeita:\n                return self.executar(s, {\'acao\': \'rejeitar\', \'movieId\': i})\n            if i not in s.assistidos:\n                s.assistidos.append(i)\n            return self.resposta(s, \'Adicionei ao histórico. Para usar como referência, escolha “Considerar”.\' + self.atualizar(s))\n        if intencao == \'explicar\':\n            alvo = valor\n            if alvo in (\'esse\', \'esse filme\', \'isso\'):\n                return self.resposta(s, \'Indique o título ou a posição (1 a 10), por exemplo: “Por que 1?”.\')\n            rec = [f for n, f in enumerate(s.recomendacoes, 1) if alvo == str(n) or normalizar(f[\'titulo\']) == alvo or re.sub(r\'\\s*\\(\\d{4}\\)$\', \'\', normalizar(f[\'titulo\'])) == alvo]\n            return self.resposta(s, rec[0][\'titulo\'] + \': \' + rec[0][\'motivo\'] if len(rec) == 1 else \'Indique o título com ano ou a posição na lista atual, como “Por que 1?”.\')\n        return self.resposta(s, \'Não entendi esse comando no modo local. Tente “Quero mais suspense”, “Não gosto de terror”, “Prefiro ficção científica”, “Por que 1?” ou “Limpar minhas preferências”. Para escolher aquele outro filme, use “Considerar” na lista. Mantive seu perfil e suas sugestões.\')\n', encoding="utf-8")


## 2. Baixar a base, preparar e avaliar
20% de teste por usuário, seed 42; limites de elegibilidade calculados apenas no treino. Notas de teste >=4 são relevantes. Itens relevantes inelegíveis continuam no denominador.

In [ ]:
from recomendador import preparar, Recomendador
treino, teste, filmes, stats = preparar()

## 3. Conferir separação e métricas

In [ ]:
import json
pares_treino = set(zip(treino.userId, treino.movieId))
pares_teste = set(zip(teste.userId, teste.movieId))
assert not (pares_treino & pares_teste)
relatorio = json.loads(Path("resultados/relatorio.json").read_text())
import pandas as pd
display(pd.DataFrame(relatorio["metricas"]).T)
print("Os valores medem recuperação por histórico, não compreensão de texto.")

## 4. Confirmar filmes e refinar o perfil
Execute a busca, revise títulos/anos e informe somente os IDs desejados. De um a cinco filmes distintos viram referências; todos os assistidos são excluídos. Mais de cinco usam os primeiros cinco. As referências são provisórias: assistir não significa gostar. A rejeição remove a contribuição do filme e mantém seu histórico. Não avaliamos offline este perfil nem a conversa com as métricas dos baselines.


In [ ]:
from conversa import Conversa, Sessao
modelo = Recomendador(filmes, stats, treino)
conversa, sessao = Conversa(modelo), Sessao()
busca = conversa.executar(sessao, {"acao": "buscar", "texto": input("Filmes assistidos (separe por ;): ")})
for item in busca["buscas"]:
    print("Busca:", item["consulta"])
    for filme in item["opcoes"]:
        print(filme["movieId"], filme["titulo"], filme["generos"])
    if not item["opcoes"]:
        print("Não encontrado; tente o título em inglês ou corrija a busca.")
selecao = input("Revise o ano. IDs dos filmes assistidos (separe por vírgula): ")
ids = [int(x.strip()) for x in selecao.split(",") if x.strip()]
if ids:
    resposta = conversa.executar(sessao, {"acao": "confirmar", "ids": ids})
    print(resposta["mensagem"])
    print("Referências:", [f["titulo"] for f in resposta["referencias"]])
    print("Fora do perfil:", [f["titulo"] for f in resposta["excedentes"]])
    display(__import__("pandas").DataFrame(resposta["recomendacoes"]))
else:
    print("Selecione pelo menos um título válido; execute a busca novamente para corrigir.")


### Refinar sem reenviar os filmes
Reexecute a próxima célula para conversar. Exemplos: “Quero mais suspense”, “Não gosto de terror”, “Por que 1?”, “Assisti TÍTULO (ANO), mas não gostei” e “Limpar minhas preferências”.


In [ ]:
resposta = conversa.executar(sessao, {"texto": input("Mensagem: ")})
print(resposta["mensagem"])
print("Preferências:", resposta["preferencias"])
display(__import__("pandas").DataFrame(resposta["recomendacoes"]))
# Para considerar um excedente, use os IDs exibidos na busca:
# conversa.executar(sessao, {"acao":"considerar", "movieId": ID_EXCEDENTE, "substituir": ID_ATIVO})
# Com cinco referências, substituir é obrigatório; sem escolha nada é substituído.


## 5. Exportar resultados no Colab
Os CSVs incluem notas dos usuários e devem ser tratados conforme os termos do MovieLens. O ZIP é para integração do grupo. A página web está no pacote de código entregue separadamente.

In [ ]:
import shutil
shutil.make_archive("resultados_movielens", "zip", "resultados")
try:
    from google.colab import files
    files.download("resultados_movielens.zip")
except ImportError:
    print("Resultados disponíveis em resultados_movielens.zip")

## Limites e referência
A API web é executada localmente com app.py. A base contém filmes, sem disponibilidade por streaming. Não foram implementados KNN, fatoração ou LLM.

Fonte, licença e citação oficial: https://files.grouplens.org/datasets/movielens/ml-latest-small-README.html

HARPER, F. M.; KONSTAN, J. A. The MovieLens Datasets: History and Context. ACM TiiS, 5(4), art. 19, 2015. https://doi.org/10.1145/2827872